In [1]:
import pandas as pd
import numpy as np

unplanned_visits = pd.read_csv(
    'Unplanned_Hospital_Visits-Hospital.csv',
    dtype={'Facility ID': str},
    low_memory=False
)

print(f"Raw shape: {unplanned_visits.shape}")
unplanned_visits.head()

Raw shape: (67046, 20)


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Denominator,Score,Lower Estimate,Higher Estimate,Number of Patients,Number of Patients Returned,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,EDAC_30_AMI,Hospital return days for heart attack patients,Fewer Days Than Average per 100 Discharges,273,-15.6,-25.9,-4.2,264,68,NaN,07/01/2021,06/30/2024
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,EDAC_30_HF,Hospital return days for heart failure patients,Average Days per 100 Discharges,652,-1.1,-10,7.9,541,186,NaN,07/01/2021,06/30/2024
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,EDAC_30_PN,Hospital return days for pneumonia patients,More Days Than Average per 100 Discharges,507,17.4,8.2,27.5,471,114,NaN,07/01/2021,06/30/2024
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Hybrid_HWR,Hybrid Hospital-Wide All-Cause Readmission Mea...,No Different Than the National Rate,2824,15.1,13.2,17.2,Not Applicable,Not Applicable,NaN,07/01/2023,06/30/2024
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,OP_32,Rate of unplanned hospital visits after colono...,No Different Than the National Rate,234,12.7,9.5,17,Not Applicable,Not Applicable,NaN,01/01/2022,12/31/2024


In [2]:
print("--- Raw nulls ---")
print(unplanned_visits.isnull().sum())

print("\n--- Raw dtypes ---")
print(unplanned_visits.dtypes)

--- Raw nulls ---
Facility ID                        0
Facility Name                      0
Address                            0
City/Town                          0
State                              0
ZIP Code                           0
County/Parish                      0
Telephone Number                   0
Measure ID                         0
Measure Name                       0
Compared to National               0
Denominator                        0
Score                              0
Lower Estimate                     0
Higher Estimate                    0
Number of Patients                 0
Number of Patients Returned        0
Footnote                       34544
Start Date                         0
End Date                           0
dtype: int64

--- Raw dtypes ---
Facility ID                    object
Facility Name                  object
Address                        object
City/Town                      object
State                          object
ZIP Code           

In [3]:
unplanned_visits.columns = (
    unplanned_visits.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace(r'[^\w]', '_', regex=True)
)

print(unplanned_visits.columns.tolist())

['facility_id', 'facility_name', 'address', 'city_town', 'state', 'zip_code', 'county_parish', 'telephone_number', 'measure_id', 'measure_name', 'compared_to_national', 'denominator', 'score', 'lower_estimate', 'higher_estimate', 'number_of_patients', 'number_of_patients_returned', 'footnote', 'start_date', 'end_date']


In [4]:
CMS_NULLS = {
    'Not Available': np.nan,
    'Not Applicable': np.nan,
    'Too Few to Report': np.nan,
    'N/A': np.nan
}

unplanned_visits = unplanned_visits.replace(CMS_NULLS)

print("CMS placeholders replaced ")
print(f"Nulls after replace:\n{unplanned_visits.isnull().sum()}")

CMS placeholders replaced 
Nulls after replace:
facility_id                        0
facility_name                      0
address                            0
city_town                          0
state                              0
zip_code                           0
county_parish                      0
telephone_number                   0
measure_id                         0
measure_name                       0
compared_to_national           15844
denominator                    31766
score                          31766
lower_estimate                 31766
higher_estimate                31766
number_of_patients             58698
number_of_patients_returned    58698
footnote                       34544
start_date                         0
end_date                           0
dtype: int64


In [5]:
unplanned_visits['facility_id'] = (
    unplanned_visits['facility_id']
    .str.strip()
    .str.zfill(6)
)

print("facility_id sample:", unplanned_visits['facility_id'].head().tolist())

facility_id sample: ['010001', '010001', '010001', '010001', '010001']


In [6]:
numeric_cols = ['score', 'denominator', 'lower_estimate', 'higher_estimate']

for col in numeric_cols:
    unplanned_visits[col] = pd.to_numeric(unplanned_visits[col], errors='coerce')

print("Numeric dtypes after cast:")
print(unplanned_visits[numeric_cols].dtypes)

Numeric dtypes after cast:
score              float64
denominator        float64
lower_estimate     float64
higher_estimate    float64
dtype: object


In [7]:
unplanned_visits['is_suppressed'] = unplanned_visits['footnote'].notna()

print("Suppression breakdown:")
print(unplanned_visits['is_suppressed'].value_counts())

Suppression breakdown:
is_suppressed
False    34544
True     32502
Name: count, dtype: int64


In [8]:
unplanned_visits['compared_to_national'] = (
    unplanned_visits['compared_to_national']
    .str.strip()
    .str.lower() 
)

print("compared_to_national values:")
print(unplanned_visits['compared_to_national'].value_counts(dropna=False))

compared_to_national values:
compared_to_national
no different than the national rate           23457
number of cases too small                     15922
NaN                                           15844
average days per 100 discharges                3720
more days than average per 100 discharges      2837
no different than expected                     2590
fewer days than average per 100 discharges     1791
worse than the national rate                    424
better than the national rate                   326
better than expected                             87
worse than expected                              48
Name: count, dtype: int64


In [9]:
cols_to_drop = [
    'facility_name', 'address', 'city_town',
    'state', 'zip_code', 'county_parish', 'telephone_number',
    'start_date', 'end_date',  
    'footnote'                 
]

unplanned_visits = unplanned_visits.drop(columns=cols_to_drop, errors='ignore')

print(f"Shape after dropping: {unplanned_visits.shape}")
print(unplanned_visits.columns.tolist())

Shape after dropping: (67046, 11)
['facility_id', 'measure_id', 'measure_name', 'compared_to_national', 'denominator', 'score', 'lower_estimate', 'higher_estimate', 'number_of_patients', 'number_of_patients_returned', 'is_suppressed']


In [10]:
dupes = unplanned_visits.duplicated(subset=['facility_id', 'measure_id']).sum()
print(f"True duplicates on composite key: {dupes}")

True duplicates on composite key: 0


In [11]:
print(f"Final shape      : {unplanned_visits.shape}")
print(f"Unique hospitals : {unplanned_visits['facility_id'].nunique()}")
print(f"Unique measures  : {unplanned_visits['measure_id'].nunique()}")

print("\n--- Nulls ---")
print(unplanned_visits.isnull().sum())

print("\n--- Measures available ---")
print(unplanned_visits['measure_id'].value_counts())

Final shape      : (67046, 11)
Unique hospitals : 4789
Unique measures  : 14

--- Nulls ---
facility_id                        0
measure_id                         0
measure_name                       0
compared_to_national           15844
denominator                    31766
score                          31766
lower_estimate                 31766
higher_estimate                31766
number_of_patients             58698
number_of_patients_returned    58698
is_suppressed                      0
dtype: int64

--- Measures available ---
measure_id
EDAC_30_AMI          4789
EDAC_30_HF           4789
EDAC_30_PN           4789
Hybrid_HWR           4789
OP_32                4789
OP_35_ADM            4789
OP_35_ED             4789
OP_36                4789
READM_30_AMI         4789
READM_30_CABG        4789
READM_30_COPD        4789
READM_30_HF          4789
READM_30_HIP_KNEE    4789
READM_30_PN          4789
Name: count, dtype: int64


In [12]:
comp_scores = unplanned_visits.pivot_table(
    index='facility_id',
    columns='measure_id',
    values='score',
    aggfunc='first'
).reset_index()

comp_scores.columns = (
    ['facility_id'] +
    ['unplanned_score_' + col.lower() for col in comp_scores.columns[1:]]
)


comp_flags = unplanned_visits.pivot_table(
    index='facility_id',
    columns='measure_id',
    values='compared_to_national',
    aggfunc='first'
).reset_index()

comp_flags.columns = (
    ['facility_id'] +
    ['unplanned_flag_' + col.lower() for col in comp_flags.columns[1:]]
)

unplanned_wide = comp_scores.merge(comp_flags, on='facility_id', how='left')

print(f"Wide table shape: {unplanned_wide.shape}")
unplanned_wide.head()

Wide table shape: (4310, 29)


,facility_id,unplanned_score_edac_30_ami,unplanned_score_edac_30_hf,unplanned_score_edac_30_pn,unplanned_score_hybrid_hwr,unplanned_score_op_32,unplanned_score_op_35_adm,unplanned_score_op_35_ed,unplanned_score_op_36,unplanned_score_readm_30_ami,...,unplanned_flag_op_32,unplanned_flag_op_35_adm,unplanned_flag_op_35_ed,unplanned_flag_op_36,unplanned_flag_readm_30_ami,unplanned_flag_readm_30_cabg,unplanned_flag_readm_30_copd,unplanned_flag_readm_30_hf,unplanned_flag_readm_30_hip_knee,unplanned_flag_readm_30_pn
0,010001,-15.6,-1.1,17.4,15.1,12.7,10.0,5.1,1.1,13.0,...,no different than the national rate,no different than the national rate,no different than the national rate,no different than expected,no different than the national rate,no different than the national rate,no different than the national rate,no different than the national rate,no different than the national rate,no different than the national rate
1,010005,NaN,12.2,-17.2,13.3,13.0,9.3,4.8,0.9,NaN,...,no different than the national rate,no different than the national rate,no different than the national rate,no different than expected,number of cases too small,NaN,no different than the national rate,no different than the national rate,no different than the national rate,no different than the national rate
2,010006,-18.0,-4.7,-0.9,15.9,11.1,NaN,NaN,1.2,12.5,...,no different than the national rate,number of cases too small,number of cases too small,no different than expected,no different than the national rate,no different than the national rate,no different than the national rate,no different than the national rate,no different than the national rate,no different than the national rate
3,010007,NaN,77.9,29.8,15.4,12.5,NaN,NaN,NaN,NaN,...,no different than the national rate,number of cases too small,number of cases too small,number of cases too small,number of cases too small,NaN,no different than the national rate,no different than the national rate,number of cases too small,no different than the national rate
4,010008,NaN,NaN,-8.9,14.7,12.7,NaN,NaN,NaN,NaN,...,no different than the national rate,NaN,NaN,number of cases too small,number of cases too small,NaN,number of cases too small,number of cases too small,NaN,no different than the national rate


In [13]:
unplanned_wide.to_csv('unplanned_visits_cleaned.csv', index=False)
print("Saved as unplanned_visits_cleaned.csv")

Saved as unplanned_visits_cleaned.csv
